# <font color='blue'> Appendix A6: Direct Preference Optimization (DPO) </font>

Reinforcement Learning from Human Feedback (RLHF) represented a major breakthrough in aligning language models with human preferences.

However,

the RLHF pipeline is complex.

It requires

- a Reward Model,
- a Value Network,
- Proximal Policy Optimization (PPO),
- careful hyperparameter tuning.

Direct Preference Optimization (DPO) simplifies this process dramatically.

Instead of first learning a Reward Model and then optimizing the policy using Reinforcement Learning,

DPO optimizes the language model **directly from human preference comparisons**.

Remarkably,

this produces behaviour similar to RLHF while requiring substantially less computational effort.

---



# <font color='orange'> 1. Motivation </font>

Suppose humans compare

two responses

to the same prompt.

```
Prompt

↓

Response A

↓

Response B

↓

Preferred Response
```

RLHF would

1. train a Reward Model,
2. estimate rewards,
3. run PPO.

DPO instead asks

> Why not train the language model directly from these comparisons?

---



# <font color='orange'> 2. The RLHF Pipeline </font>

Classical RLHF

```
Pretraining

↓

SFT

↓

Human Preferences

↓

Reward Model

↓

PPO

↓

Aligned Model
```

This pipeline involves several separate optimization problems.

---



# <font color='orange'> 3. The DPO Pipeline </font>

DPO simplifies the process.

```
Pretraining

↓

SFT

↓

Human Preferences

↓

Direct Optimization

↓

Aligned Model
```

No Reward Model is trained.

No Reinforcement Learning algorithm is required.

---



# <font color='orange'> 4. Preference Dataset </font>

As before,

the dataset contains

$$
\boxed{
(x,y_w,y_l),
}
$$

where

- \(x\) is the prompt,
- \(y_w\) is the preferred response,
- \(y_l\) is the rejected response.

Exactly the same preference data used for RLHF can be used for DPO.

---



# <font color='orange'> 5. Key Insight </font>

Suppose

the language model assigns

probabilities

to both responses.

```
Preferred Response

↓

High Probability
```

```
Rejected Response

↓

Low Probability
```

Rather than predicting

a reward,

DPO simply encourages

the preferred response

to become

more probable

than the rejected response.

---



# <font color='orange'> 6. Reference Model </font>

DPO keeps

the original

Supervised Fine-Tuned model

as a

**reference model**.

```
Reference Model

↓

Fixed
```

```
Current Model

↓

Trainable
```

The reference model prevents

the updated model

from drifting too far

from its original behaviour.

Notice that this plays a role similar to the KL-divergence penalty in RLHF.

---



# <font color='orange'> 7. Relative Preference </font>

Suppose

the current model assigns

$$
\pi_\theta(y_w|x)
$$

to the preferred response,

and

$$
\pi_\theta(y_l|x)
$$

to the rejected response.

If

the preferred response

already has much higher probability,

very little learning is required.

Otherwise,

the optimization increases

its probability.

---



# <font color='orange'> 8. The DPO Loss </font>

The optimization objective is

$$
\boxed{
L_{\mathrm{DPO}}
=
-
\log
\sigma
\left(
\beta
\left[
\log
\frac{
\pi_\theta(y_w|x)
}{
\pi_\theta(y_l|x)
}
-
\log
\frac{
\pi_{\mathrm{ref}}(y_w|x)
}{
\pi_{\mathrm{ref}}(y_l|x)
}
\right]
\right).
}
$$

Here,

- \(\sigma\) is the sigmoid function,
- \(\beta\) controls the optimization strength.

This objective directly encourages

preferred responses

to become more likely

than rejected responses.

---



# <font color='orange'> 9. Intuition Behind the Loss </font>

The loss compares

two probability ratios.

The first ratio

comes from

the current model.

The second ratio

comes from

the reference model.

If

the current model

already prefers

the human-preferred response

more strongly

than the reference model,

the loss becomes small.

Otherwise,

gradient descent

adjusts the model accordingly.

---



# <font color='orange'> 10. Why No Reward Model? </font>

One of the major theoretical results behind DPO is that,

under appropriate assumptions,

the optimal policy in RLHF can be written directly in terms of preference probabilities.

Instead of explicitly learning

a Reward Model,

its effect is absorbed

into the optimization objective.

Thus,

preference optimization becomes

a supervised learning problem.

---



# <font color='orange'> 11. Training Pipeline </font>

The complete workflow becomes

```
Prompt

↓

Generate Responses

↓

Human Preferences

↓

Compute DPO Loss

↓

Gradient Descent

↓

Updated Language Model
```

The pipeline is considerably simpler than RLHF.

---



# <font color='orange'> 12. Advantages </font>

Compared with PPO-based RLHF,

DPO

- requires fewer models,
- eliminates Reward Model training,
- eliminates PPO,
- is simpler to implement,
- is computationally cheaper,
- is generally more stable.

These properties explain its rapid adoption.

---



# <font color='orange'> 13. Limitations </font>

DPO still depends on

high-quality preference data.

Furthermore,

it assumes

that the preference comparisons adequately capture

the behaviours we wish the model to learn.

Like all alignment methods,

its performance depends on

the quality and diversity of the training data.

---

# <font color='red'> 14. Mathematical Foundations </font>

Suppose

the preference dataset contains

$$
(x,y_w,y_l).
$$

The current model defines

$$
\boxed{
\pi_\theta(y|x).
}
$$

The fixed reference model defines

$$
\boxed{
\pi_{\mathrm{ref}}(y|x).
}
$$

The DPO objective is

$$
\boxed{
L_{\mathrm{DPO}}
=
-
\log
\sigma
\left(
\beta
\left[
\log
\frac{
\pi_\theta(y_w|x)
}{
\pi_\theta(y_l|x)
}
-
\log
\frac{
\pi_{\mathrm{ref}}(y_w|x)
}{
\pi_{\mathrm{ref}}(y_l|x)
}
\right]
\right).
}
$$

Interpretation:

- Increase the probability of preferred responses.
- Decrease the probability of rejected responses.
- Stay close to the reference model.
- Learn directly from preference comparisons.

Unlike RLHF,

there is no explicit reward prediction or policy optimization step.

---



# <font color='orange'> 15. RLHF vs DPO </font>

| RLHF | DPO |
|:---|:---|
| Reward Model | No Reward Model |
| PPO Optimization | Direct Optimization |
| Value Network | Not Required |
| Reinforcement Learning | Supervised Optimization |
| Computationally Expensive | Simpler and More Efficient |

Both methods aim to align the model with human preferences, but DPO achieves this through a much simpler optimization procedure.

---



# <font color='orange'> 16. Common Misconceptions </font>

### Misconception 1

> DPO requires a Reward Model.

**False.**

DPO operates directly on preference comparisons. It does not train or optimize a separate Reward Model.

---

### Misconception 2

> DPO is no longer related to Reinforcement Learning.

**Not entirely.**

DPO was derived by analyzing the optimization problem underlying RLHF. Although it does not perform reinforcement learning during training, its objective is closely connected to the RLHF formulation.

---

### Misconception 3

> DPO guarantees perfect alignment.

**False.**

Like all alignment methods, DPO depends on the quality, diversity, and representativeness of the human preference data. Poor preference data can still lead to undesirable behaviour.

---

# <font color='purple'> 17. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Direct Preference Optimization | Aligns models directly from preference data |
| Preference Dataset | Prompt with preferred and rejected responses |
| Reference Model | Fixed SFT model used as a baseline |
| Probability Ratio | Compares preferred and rejected responses |
| DPO Loss | Encourages preferred responses while remaining close to the reference model |
| No Reward Model | Eliminates a separate reward-learning stage |
| No PPO | Avoids reinforcement learning optimization |

> **Key Insight:** Direct Preference Optimization reformulates the alignment problem so that language models can be trained directly from human preference comparisons without explicitly training a Reward Model or running reinforcement learning. By increasing the probability of preferred responses relative to rejected ones while remaining close to a fixed reference model, DPO achieves many of the benefits of RLHF with a significantly simpler and more computationally efficient training procedure. This elegance has made DPO and its variants central techniques in the alignment of many modern large language models.